# Laboratorio #7 - Aprendizaje por Refuerzo

* Paula Barillas - 22764
* Gerardo Pineda - 22880
* Mónica Salvatierra - 22249
* Bianca Calderón - 22272
* Link del repositorio: https://github.com/paulabaal12/LAB7-RL.git

---

## Contexto del problema

Una empresa de ciberseguridad opera un **Sistema de Detección de Intrusiones (IDS)** para redes corporativas. El sistema observa el tráfico en tiempo real y debe elegir, en cada instante, una de cuatro acciones:

| Acción | Descripción | Nivel de disrupción |
|---|---|---|
| `IGNORAR` | No hacer nada | Nulo |
| `ALERTAR` | Notificar al equipo de seguridad | Bajo |
| `BLOQUEAR_TEMPORAL` | Bloquear temporalmente el tráfico sospechoso | Medio |
| `AISLAR_SEGMENTO` | Aislar completamente el segmento de red afectado | Alto |

El estado $s_t \in \mathbb{R}^{256}$ combina métricas de tráfico (bytes/paquetes por segundo, entropía de puertos, etc.), patrones de comportamiento de usuarios y resúmenes de logs recientes.

Se pide evaluar de forma **crítica** la propuesta de usar **DQN estándar** antes de implementar nada.


---
# Task 1

## Task 1.1.a

**DQN es apropiado a nivel de algoritmo**, pero **no en su arquitectura original**.

### Por qué la arquitectura "clásica" de DQN (CNN) no aplica aquí

El DQN de Mnih et al. (2015) usa **redes convolucionales (CNN)** porque su entrada son *frames* de píxeles (84×84×4), donde:

- Existe **correlación espacial local**: píxeles vecinos están relacionados (bordes, texturas, objetos).
- Existe **invarianza traslacional**: un mismo patrón (p. ej. la pelota en Pong) puede aparecer en cualquier posición y debe reconocerse igual.
- Los filtros convolucionales explotan estas dos propiedades para compartir parámetros y reducir dimensionalidad de forma eficiente.

**Ninguna de esas dos propiedades existe en nuestro vector de estado.** Las 256 dimensiones son *features* heterogéneas y no ordenadas espacialmente (p. ej. la dimensión 3 puede ser "bytes/seg", la dimensión 47 "entropía de puertos destino", la dimensión 130 "número de logins fallidos"). No hay ninguna razón para asumir que la dimensión $i$ está relacionada con la dimensión $i+1$ de la misma forma en que un píxel se relaciona con su vecino. Aplicar convoluciones aquí impondría un sesgo inductivo (localidad + compartición de pesos) que **no corresponde a la estructura real del dato**, y en el mejor de los casos sería equivalente a una capa densa mal condicionada; en el peor, degradaría el aprendizaje.

### Arquitectura recomendada: MLP (red totalmente conectada)

Para un vector de estado numérico y continuo, el aproximador natural de $Q(s,a;\mathbf{w})$ es un **Perceptrón Multicapa (MLP / fully-connected feedforward network)**:

$$
Q(s,\cdot;\mathbf{w}): \mathbb{R}^{256} \;\rightarrow\; \mathbb{R}^{4}
$$

es decir, una red que recibe el vector de 256 dimensiones y produce **un valor Q por cada una de las 4 acciones** en una sola pasada hacia adelante (arquitectura estándar de DQN: una salida por acción, no una red por acción).

propuesta:

- **Capa de entrada:** 256 unidades (una por feature).
- **Normalización de entrada:** `BatchNorm`/`LayerNorm` o estandarización previa (z-score), ya que las métricas de tráfico tienen escalas muy distintas (bytes/seg vs. conteos vs. proporciones en [0,1]); sin esto, el gradiente estará dominado por las features de mayor magnitud.
- **2–3 capas ocultas densas** (p. ej. 256 → 128 → 64), con activación **ReLU**.
- **Capa de salida:** 4 unidades lineales (una por acción), sin activación, porque $Q$ puede ser positivo o negativo.
- Opcionalmente, arquitectura **Dueling** (ver 1.1.b): dos cabezas ($V(s)$ y $A(s,a)$) que se combinan al final — esto sigue siendo un MLP, solo cambia cómo se combinan las últimas capas.



In [ ]:
import numpy as np

# Implementacion ilustrativa en NumPy puro del aproximador MLP de Q(s,a;w)

class QNetworkIDS:
    '''Aproximador MLP de Q(s,a;w) para el IDS.
    Entrada: vector de estado de 256 dimensiones continuas.
    Salida: 4 valores Q, uno por accion discreta.
    Arquitectura: 256 -> 128 -> 64 -> 4, con LayerNorm en la entrada y ReLU oculta.
    '''
    def __init__(self, state_dim=256, n_actions=4, hidden=(256, 128, 64), seed=0):
        rng = np.random.default_rng(seed)
        dims = [state_dim] + list(hidden) + [n_actions]
        self.W = [rng.normal(0, np.sqrt(2 / dims[i]), size=(dims[i], dims[i + 1])) for i in range(len(dims) - 1)]
        self.b = [np.zeros(dims[i + 1]) for i in range(len(dims) - 1)]
        self.n_layers = len(self.W)

    @staticmethod
    def _layer_norm(x, eps=1e-5):
        mu = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        return (x - mu) / np.sqrt(var + eps)

    @staticmethod
    def _relu(x):
        return np.maximum(0, x)

    def forward(self, state):
        x = self._layer_norm(state)                 # normaliza escalas heterogeneas de las 256 features
        for i in range(self.n_layers - 1):
            x = self._relu(x @ self.W[i] + self.b[i])
        q = x @ self.W[-1] + self.b[-1]              # capa de salida lineal: Q puede ser + o -
        return q                                     # (batch, 4) -> Q(s,IGNORAR..AISLAR)


# Verificacion rapida de formas (sanity check arquitectonico)
net = QNetworkIDS()
dummy_state = np.random.default_rng(1).normal(size=(8, 256))  # batch de 8 estados de 256 dims
q_values = net.forward(dummy_state)

print("Forma de entrada  :", dummy_state.shape)
print("Forma de salida   :", q_values.shape, "-> (batch, n_acciones)")
n_params = sum(w.size for w in net.W) + sum(b.size for b in net.b)
print("Parametros totales:", n_params)
print("\nEjemplo de Q-values para el primer estado del batch:")
acciones = ["IGNORAR", "ALERTAR", "BLOQUEAR_TEMPORAL", "AISLAR_SEGMENTO"]
for a, q in zip(acciones, q_values[0]):
    print(f"  Q(s, {a:<18s}) = {q: .4f}")
print(f"  -> accion elegida (argmax): {acciones[int(np.argmax(q_values[0]))]}")


Forma de entrada  : (8, 256)
Forma de salida   : (8, 4) -> (batch, n_acciones)
Parametros totales: 107204

Ejemplo de Q-values para el primer estado del batch:
  Q(s, IGNORAR           ) = -0.6030
  Q(s, ALERTAR           ) =  0.4614
  Q(s, BLOQUEAR_TEMPORAL ) =  2.4254
  Q(s, AISLAR_SEGMENTO   ) =  0.0172
  -> accion elegida (argmax): BLOQUEAR_TEMPORAL


---
## Task 1.1.b

**Sí**

DQN (y Q-learning en general) requiere que, para calcular $\max_{a'} Q(s',a';\mathbf{w})$ en el objetivo de Bellman, se pueda **enumerar y evaluar todas las acciones posibles**. Con un espacio discreto y pequeño (4 acciones: `IGNORAR`, `ALERTAR`, `BLOQUEAR_TEMPORAL`, `AISLAR_SEGMENTO`), esto es trivial: la red produce las 4 salidas en un solo forward pass y el `argmax`/`max` se calcula en $O(4)$.

Esto **contrasta directamente** con dominios de acción continua (p. ej. control de un robot con torques continuos), donde $\max_{a'} Q(s',a')$ requiere una optimización interna costosa o intratable, y por eso ahí sí se necesitan algoritmos distintos como **DDPG, TD3 o SAC** (actor-crítico con política parametrizada que aproxima directamente el argmax). **No es nuestro caso.**

### Qué variante conviene.

1. **Double DQN (DDQN):** DQN estándar sufre **sobreestimación sistemática** de $Q$ porque usa la misma red para *seleccionar* y *evaluar* la acción máxima ($\max_{a'} Q(s',a';\mathbf{w})$ está sesgado hacia arriba por el ruido de estimación). En un IDS esto es peligroso: una sobreestimación del valor de `IGNORAR` en estados ambiguos podría hacer que el agente subestime el riesgo de una intrusión real. DDQN desacopla selección y evaluación:
 $$y = r + \gamma\, Q\big(s', \arg\max_{a'} Q(s',a';\mathbf{w});\ \mathbf{w}^-\big)$$
 usando la red *online* para elegir la acción y la red *target* ($\mathbf{w}^-$) para evaluarla.

2. **Dueling DQN:** con solo 4 acciones, es común que en muchos estados **la acción óptima no dependa de las diferencias finas entre acciones**, sino del valor general del estado (p. ej. "este estado es benigno, cualquier acción pasiva está bien" vs. "este estado es crítico"). Dueling DQN separa:
 $$Q(s,a) = V(s) + \Big(A(s,a) - \frac{1}{|\mathcal{A}|}\sum_{a'} A(s,a')\Big)$$
 lo que permite aprender $V(s)$ de forma más eficiente (compartiendo señal entre las 4 acciones) sin necesitar experimentar cada acción individualmente en cada estado — muy útil cuando ciertas acciones (`AISLAR_SEGMENTO`) se toman con muy poca frecuencia.

3. Dado el desbalance de clases, también se recomienda **Prioritized Experience Replay (PER)**, que no cambia el espacio de acciones pero sí la forma de entrenar.

**Conclusión:** el número y tipo de acciones (4, discretas, pequeñas) **no exige cambiar de familia de algoritmo**; DQN es adecuado. Lo que se recomienda es usar una variante **Double + Dueling DQN**, no por el tamaño del espacio de acciones en sí, sino por los sesgos de estimación y el desbalance de datos propios de este dominio.


---
## Task 1.1.c

### Formalización del problema

Sea $\mathcal{D} = \{(s_i, a_i, r_i, s_i')\}_{i=1}^{N}$ el buffer de experience replay con capacidad $N$, y sea $p_{\text{crit}} < 0.001$ la probabilidad de que una transición cualquiera corresponda a un evento crítico (intrusión real). Bajo muestreo **uniforme**, cada transición tiene probabilidad $1/N$ de ser elegida en cada extracción, independientemente de su contenido informativo.

El número esperado de transiciones críticas dentro del buffer es:

$$
\mathbb{E}[\#\text{críticas en } \mathcal{D}] = N \cdot p_{\text{crit}} < 0.001\,N
$$

y la probabilidad de que un **minibatch** de tamaño $B$ (muestreado uniformemente y sin reemplazo del buffer) contenga **al menos una** transición crítica es aproximadamente (para $N$ grande):

$$
P(\text{al menos 1 crítica en el batch}) \approx 1 - (1-p_{\text{crit}})^{B}
$$

Con $p_{\text{crit}} = 0.001$ y un batch típico $B=64$: $P \approx 1-(0.999)^{64} \approx 6.2\%$. Es decir, **más del 93% de los batches de entrenamiento no contienen ningún evento crítico**, y cuando aparece uno, es prácticamente uno solo entre 64 muestras dominadas por tráfico normal.

### Consecuencias formales sobre el aprendizaje

1. **Dilución del gradiente:** el error cuadrático medio que se minimiza,
 $$\mathcal{L}(\mathbf{w}) = \mathbb{E}_{(s,a,r,s')\sim U(\mathcal{D})}\Big[\big(y - Q(s,a;\mathbf{w})\big)^2\Big],$$
 al tomarse en esperanza bajo la distribución **empírica** del buffer (que hereda el desbalance real del entorno), pondera implícitamente el error de los estados normales muxho más que el de los estados críticos. El gradiente promedio del batch está dominado casi en su totalidad por transiciones benignas.

2. **Olvido catastrófico local:** incluso si una experiencia crítica logra actualizar la red una vez, cientos de actualizaciones posteriores basadas en tráfico normal pueden "borrar" ese ajuste (fenómeno de *catastrophic forgetting*), ya que no hay mecanismo que refuerce revisitar esa experiencia con mayor frecuencia.

3. **Sub-representación estructural (no solo estadística):** este es el problema clásico de **desbalance de clases** trasladado a RL: el buffer, al llenarse con la distribución natural del entorno, es un espejo de esa rareza. El muestreo uniforme asume implícitamente que todas las transiciones son igualmente informativas para el aprendizaje, lo cual es falso: las transiciones raras y las de alto error de predicción (TD-error) suelen ser las más informativas.

4. **Riesgo de "olvido" total en buffers de capacidad fija (FIFO):** si $N$ es finito y el buffer se sobrescribe circularmente, una racha larga sin intrusiones puede **expulsar por completo** las pocas experiencias críticas que existían, dejando al agente sin ningún ejemplo de ese caso en memoria.

### Solución: Prioritized Experience Replay (PER)

**PER (Schaul et al., 2016)** resuelve esto reemplazando el muestreo uniforme por un muestreo **proporcional a la magnitud del TD-error**, que actúa como *proxy* de "qué tan sorprendente/informativa" es una transición:

$$
P(i) = \frac{p_i^{\alpha}}{\sum_k p_k^{\alpha}}, \qquad p_i = |\delta_i| + \epsilon
$$

donde $\delta_i = y_i - Q(s_i,a_i;\mathbf{w})$ es el TD-error, $\alpha$ controla cuánta prioridad se usa ($\alpha=0$ recupera muestreo uniforme), y $\epsilon>0$ evita probabilidad cero.

**Resuelve porque:** los eventos de intrusión real, al ser poco frecuentes, son también los que la red **peor predice inicialmente** (alto $|\delta_i|$), porque no ha tenido suficientes ejemplos para aprenderlos. PER los muestreará con mayor frecuencia que su frecuencia natural en el entorno, forzando a la red a prestarles atención desproporcionada — exactamente lo contrario del efecto de dilución descrito arriba.

In [ ]:
import numpy as np

# Ilustracion numerica del argumento formal
p_crit = 0.001     # probabilidad de evento critico por paso de tiempo
N = 100_000        # capacidad del buffer
B = 64             # tamano de minibatch

esperado_en_buffer = N * p_crit
prob_batch_con_al_menos_1 = 1 - (1 - p_crit) ** B

print(f"Transiciones criticas esperadas en el buffer (N={N}): {esperado_en_buffer:.1f}")
print(f"P(al menos 1 transicion critica en un batch de {B}) = {prob_batch_con_al_menos_1*100:.2f}%")
print(f"P(NINGUNA transicion critica en el batch)           = {(1-prob_batch_con_al_menos_1)*100:.2f}%")

# Simulacion Monte Carlo para confirmar el calculo analitico
rng = np.random.default_rng(42)
n_sim = 200_000
buffer_sim = rng.random(n_sim) < p_crit         # True = transicion critica
muestras_batches = rng.choice(n_sim, size=(20_000, B), replace=True)
contiene_critica = buffer_sim[muestras_batches].any(axis=1)

print(f"\n[Monte Carlo, {20_000} batches simulados]")
print(f"Fraccion de batches con >=1 transicion critica: {contiene_critica.mean()*100:.2f}%  (vs. formula analitica: {prob_batch_con_al_menos_1*100:.2f}%)")


Transiciones criticas esperadas en el buffer (N=100000): 100.0
P(al menos 1 transicion critica en un batch de 64) = 6.20%
P(NINGUNA transicion critica en el batch)           = 93.80%

[Monte Carlo, 20000 batches simulados]
Fraccion de batches con >=1 transicion critica: 6.40%  (vs. formula analitica: 6.20%)


---
## Task 1.1.d — Riesgo de optimizar solo falsos positivos, y diseño de recompensa multi-componente

### El comportamiento indeseable

Si la función de recompensa se diseña **únicamente** para minimizar falsos positivos (p. ej. $r = -1$ cada vez que el sistema alerta/bloquea/aísla tráfico que en realidad era benigno, y $r=0$ en cualquier otro caso), el agente encuentra una **política degenerada trivial** que maximiza esa recompensa sin resolver el problema real:

> **El agente aprende a elegir siempre `IGNORAR`.**

si nunca actúa, nunca puede generar un falso positivo. El resultado es una tasa de falsos positivos del 0% — pero también una **tasa de falsos negativos del 100%**: el sistema deja pasar todas las intrusiones reales. Esto es un caso de **reward hacking **: se optimiza perfectamente la métrica programada, mientras se ignora por completo el objetivo real (detectar y contener intrusiones), que nunca fue explícitamente premiado.


### Diseño de una función de recompensa con múltiples componentes

$$
r_t = w_1 \cdot R_{\text{detección}} + w_2 \cdot R_{\text{FN}} + w_3 \cdot R_{\text{FP}} + w_4 \cdot R_{\text{costo\_acción}}
$$

| Componente | Situación que premia/penaliza | Valor propuesto | Justificación de magnitud |
|---|---|---|---|
| **$R_{\text{detección}}$** (verdadero positivo) | El agente `BLOQUEA` o `AISLA` cuando **sí** había una intrusión real | **+10** | Debe ser claramente positivo y grande: es el objetivo central del sistema. Sin una recompensa positiva explícita por acertar, no hay incentivo para actuar en absoluto (ver comportamiento degenerado arriba). |
| **$R_{\text{FN}}$** (falso negativo) | El agente `IGNORA` o solo `ALERTA` (sin contener) cuando **sí** había una intrusión real | **−20** | Debe ser la penalización **más grande de todas**, y estrictamente mayor en magnitud que $R_{\text{FP}}$. El costo de negocio de una intrusión no contenida (robo de datos, ransomware, multas regulatorias, daño reputacional) es órdenes de magnitud mayor y a menudo irreversible, comparado con el costo de una falsa alarma. Esta asimetría 2:1 (o mayor, ajustable con datos reales de costo) es la que **evita matemáticamente** que "no hacer nada" sea la política óptima. |
| **$R_{\text{FP}}$** (falso positivo) | El agente `BLOQUEA` o `AISLA` tráfico que **era benigno** | **−2** | Penalización moderada: sí queremos desincentivar la sobre-reacción (que es el problema actual del sistema de reglas, 8%), pero no tanto como para volver a incentivar la pasividad total. Es menor que $R_{\text{FN}}$ para reflejar que el costo de negocio (interrupción temporal, fricción con usuarios) es recuperable. |
| **$R_{\text{costo\_acción}}$** (costo de disrupción operativa) | Costo proporcional a cuán invasiva es la acción, independientemente de si acertó | `IGNORAR`: 0, `ALERTAR`: −0.1, `BLOQUEAR`: −0.5, `AISLAR`: −1.0 | Penalización pequeña y **graduada** que actúa como regularizador: entre dos acciones que ambos contienen correctamente una amenaza, se prefiere la menos disruptiva (p. ej. `BLOQUEAR` en vez de `AISLAR` todo un segmento si es suficiente). Su magnitud es intencionalmente pequeña frente a $R_{\text{detección}}$/$R_{\text{FN}}$ para que **nunca** domine la decisión de actuar o no ante una amenaza real — solo afina *cuál* acción usar una vez que ya conviene intervenir. |


Con esta función de recompensa, la política óptima deja deignorar y pasa a depender genuinamente de si el estado corresponde o no a una amenaza real, penalizando de forma asimétrica los dos tipos de error posibles.


In [ ]:
def calcular_recompensa(accion, es_intrusion_real):
    '''Ejemplo simplificado de la funcion de recompensa multi-componente de 1.1.d
    accion in {"IGNORAR","ALERTAR","BLOQUEAR_TEMPORAL","AISLAR_SEGMENTO"}
    es_intrusion_real: bool -> ground truth del evento simulado
    '''
    costo_accion = {
        "IGNORAR": 0.0,
        "ALERTAR": -0.1,
        "BLOQUEAR_TEMPORAL": -0.5,
        "AISLAR_SEGMENTO": -1.0,
    }[accion]

    contuvo = accion in ("BLOQUEAR_TEMPORAL", "AISLAR_SEGMENTO")

    if es_intrusion_real and contuvo:
        r_deteccion, r_fn, r_fp = 10.0, 0.0, 0.0        # verdadero positivo
    elif es_intrusion_real and not contuvo:
        r_deteccion, r_fn, r_fp = 0.0, -20.0, 0.0       # falso negativo
    elif (not es_intrusion_real) and contuvo:
        r_deteccion, r_fn, r_fp = 0.0, 0.0, -2.0        # falso positivo
    else:
        r_deteccion, r_fn, r_fp = 0.0, 0.0, 0.0         # verdadero negativo

    return r_deteccion + r_fn + r_fp + costo_accion


# Comparacion: politica degenerada "siempre ignorar" vs. politica que reacciona correctamente
import numpy as np
rng = np.random.default_rng(0)
n_pasos = 100_000
p_intrusion = 0.001
eventos = rng.random(n_pasos) < p_intrusion

# Politica degenerada (solo penaliza FP -> aprende a ignorar siempre)
recompensa_degenerada = sum(-20.0 if e else 0.0 for e in eventos)  # bajo reward "solo-FP", esto seria 0; mostramos el costo real oculto

# Politica correcta bajo nuestra funcion multi-componente
recompensa_ignorar_siempre = sum(calcular_recompensa("IGNORAR", e) for e in eventos)
recompensa_reactiva_correcta = sum(
    calcular_recompensa("BLOQUEAR_TEMPORAL" if e else "IGNORAR", e) for e in eventos
)

print(f"Pasos simulados: {n_pasos}, intrusiones reales: {eventos.sum()}")
print(f"Retorno acumulado - politica 'siempre IGNORAR'      : {recompensa_ignorar_siempre:,.1f}")
print(f"Retorno acumulado - politica reactiva (bloquea si hay intrusion): {recompensa_reactiva_correcta:,.1f}")


Pasos simulados: 100000, intrusiones reales: 90
Retorno acumulado - politica 'siempre IGNORAR'      : -1,800.0
Retorno acumulado - politica reactiva (bloquea si hay intrusion): 855.0

-> Con la funcion de recompensa multi-componente, la politica que SI reacciona
   ante las intrusiones reales obtiene mayor retorno que la politica pasiva,
   justo lo contrario de lo que ocurriria optimizando solo falsos positivos.


## Fuentes

- Mnih et al. (2015), *"Human-level control through deep reinforcement learning"*. PDF: https://www.cs.ucf.edu/~lboloni/Teaching/CAP5636_Fall2023/homeworks/2015-Volodymyr%20Mnih-DQN.pdf

- van Hasselt, Guez & Silver (2016), (https://arxiv.org/pdf/1509.06461)

- Krakovna et al., DeepMind, *"Specification gaming: the flip side of AI ingenuity"*. El "specification gaming" es un comportamiento que satisface la especificación literal de un objetivo sin lograr el resultado pretendido; un agente de RL puede encontrar un atajo para obtener mucha recompensa sin completar la tarea tal como el diseñador humano la pretendía. Lista de ejemplos: https://www.lesswrong.com/posts/AanbbjYr5zckMKde7/specification-gaming-examples-in-ai
- Ejemplo aplicado a IDS con recompensa adaptativa: se propone un mecanismo de recompensa adaptativo con bono por detección de ataques para abordar el problema de desbalance de clases, junto con Prioritized Experience Replay — JFES, DRL-IDS con D3QN: https://alfarabiuc.edu.iq/Journal/index.php/Farabi-Eng/en/article/download/116/103/150

**Libro de referencia general** (para conceptos base): Sutton & Barto, *"Reinforcement Learning: An Introduction"* (2nd ed.).

## Prompt de apoyo


```
Serás un revisor de un trabajo de Aprendizaje por Refuerzo. Te voy a compartir un análisis sobre DQN para un sistema de detección de intrusiones (IDS), dime: si el argumento es técnicamente correcto, y si hay algo importante que se omitió.